<a href="https://colab.research.google.com/github/anberaziz5/medscript-ai/blob/main/medscript_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sat Jun 27 14:16:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformers datasets seqeval accelerate -q

In [ ]:
import json

# Standard structured mock samples matching your preprocessing and augmentation outputs
final_train_data = [
  {"tokens": ["مریض", "کو", "fever", "ہے", "۔"], "labels": ["O", "O", "B-DISEASE", "O", "O"]},
  {"tokens": ["Paracetamol", "500mg", "دیں", "۔"], "labels": ["B-DRUG", "B-DOSE", "O", "O"]},
  {"tokens": ["Patient", "has", "cough", "۔"], "labels": ["O", "O", "B-SYMPTOM", "O"]},
  {"tokens": ["Prescribe", "Amoxicillin", "250mg", "۔"], "labels": ["O", "B-DRUG", "B-DOSE", "O"]},
  {"tokens": ["Flagyl", "400mg", "oral", "۔"], "labels": ["B-DRUG", "B-DOSE", "B-ROUTE", "O"]},
  {"tokens": ["Patient", "has", "cough", "۔", "Prescribe", "Panadol", "500mg", "oral", "twice", "daily", "۔"], "labels": ["O", "O", "B-SYMPTOM", "O", "O", "B-DRUG", "B-DOSE", "B-ROUTE", "B-DOSE", "I-DOSE", "O"]},
  {"tokens": ["مریض", "کو", "fever", "ہے", "۔", "Give", "Augmentin", "625mg", "oral", "۔"], "labels": ["O", "O", "B-DISEASE", "O", "O", "O", "B-DRUG", "B-DOSE", "B-ROUTE", "O"]},
  {"tokens": ["Flagyl", "400mg", "دیں", "for", "diarrhea", "۔"], "labels": ["B-DRUG", "B-DOSE", "O", "O", "B-SYMPTOM", "O"]}
]

val_data = [
  {"tokens": ["Take", "Panadol", "daily", "۔"], "labels": ["O", "B-DRUG", "B-DOSE", "O"]}
]

test_data = [
  {"tokens": ["Calamine", "lotion", "for", "rash", "۔"], "labels": ["B-DRUG", "B-ROUTE", "O", "B-SYMPTOM", "O"]}
]

# Write them out directly to your current local storage session
with open('final_train.json', 'w', encoding='utf-8') as f:
    json.dump(final_train_data, f, ensure_ascii=False, indent=2)

with open('val.json', 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

with open('test.json', 'w', encoding='utf-8') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

print("Data splits successfully generated inside this session!")

Data splits successfully generated inside this session!


In [ ]:
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset, DatasetDict
from seqeval.metrics import f1_score, precision_score, recall_score

# ==========================================
# 1. SETUP DEFINITIONS & ENVIRONMENT GLOBAL
# ==========================================
MODEL_NAME = 'bert-base-multilingual-cased'
LABEL2ID = {
    "O": 0,
    "B-DRUG": 1,   "I-DRUG": 2,
    "B-DISEASE": 3, "I-DISEASE": 4,
    "B-SYMPTOM": 5, "I-SYMPTOM": 6,
    "B-DOSE": 7,   "I-DOSE": 8,
    "B-ROUTE": 9,  "I-ROUTE": 10,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ==========================================
# 2. GENERATE AND LOAD CLEAN SPLITS
# ==========================================
final_train_data = [
  {"tokens": ["مریض", "کو", "fever", "ہے", "۔"], "labels": ["O", "O", "B-DISEASE", "O", "O"]},
  {"tokens": ["Paracetamol", "500mg", "دیں", "۔"], "labels": ["B-DRUG", "B-DOSE", "O", "O"]},
  {"tokens": ["Patient", "has", "cough", "۔"], "labels": ["O", "O", "B-SYMPTOM", "O"]},
  {"tokens": ["Prescribe", "Amoxicillin", "250mg", "۔"], "labels": ["O", "B-DRUG", "B-DOSE", "O"]},
  {"tokens": ["Flagyl", "400mg", "oral", "۔"], "labels": ["B-DRUG", "B-DOSE", "B-ROUTE", "O"]},
  {"tokens": ["Patient", "has", "cough", "۔", "Prescribe", "Panadol", "500mg", "oral", "twice", "daily", "۔"], "labels": ["O", "O", "B-SYMPTOM", "O", "O", "B-DRUG", "B-DOSE", "B-ROUTE", "B-DOSE", "I-DOSE", "O"]},
  {"tokens": ["مریض", "کو", "fever", "ہے", "۔", "Give", "Augmentin", "625mg", "oral", "۔"], "labels": ["O", "O", "B-DISEASE", "O", "O", "O", "B-DRUG", "B-DOSE", "B-ROUTE", "O"]},
  {"tokens": ["Flagyl", "400mg", "دیں", "for", "diarrhea", "۔"], "labels": ["B-DRUG", "B-DOSE", "O", "O", "B-SYMPTOM", "O"]}
]
val_data = [{"tokens": ["Take", "Panadol", "daily", "۔"], "labels": ["O", "B-DRUG", "B-DOSE", "O"]}]
test_data = [{"tokens": ["Calamine", "lotion", "for", "rash", "۔"], "labels": ["B-DRUG", "B-ROUTE", "O", "B-SYMPTOM", "O"]}]

def convert_to_hf(json_split):
    return Dataset.from_dict({
        'tokens': [item['tokens'] for item in json_split],
        'labels': [item['labels'] for item in json_split]
    })

hf_dataset = DatasetDict({
    'train': convert_to_hf(final_train_data),
    'validation': convert_to_hf(val_data),
    'test': convert_to_hf(test_data)
})

# ==========================================
# 3. ALIGNMENT & TOKENIZATION
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True, padding='max_length', max_length=128)
    labels = []
    for i, label_list in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(LABEL2ID.get(label_list[word_id], 0))
            else:
                label_ids.append(-100)
            prev_word_id = word_id
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

tokenized_datasets = hf_dataset.map(tokenize_and_align_labels, batched=True)

# ==========================================
# 4. INITIALIZE MODEL & METRICS
# ==========================================
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABEL2ID), id2label=ID2LABEL, label2id=LABEL2ID
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=-1)
    true_predictions = [
        [ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [ID2LABEL[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

# ==========================================
# 5. RUN FINE-TUNING LOOP
# ==========================================
training_args = TrainingArguments(
    output_dir='./mbert-baseline',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy='epoch', # Fixed recent HF parameter name
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)

print("Launching unified baseline mBERT model training run...")
trainer.train()

# Evaluate final test splits
baseline_results = trainer.evaluate(tokenized_datasets['test'])
print("\n--- FINAL BASELINE EVALUATION SCORES ---")
print(f"Precision: {baseline_results['eval_precision']:.4f}")
print(f"Recall: {baseline_results['eval_recall']:.4f}")
print(f"BASELINE F1 SCORE: {baseline_results['eval_f1']:.4f}")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Training Loss,Validation Loss,Epoch,Precision,Recall,F1
No log,1.929930,3,0.000000,0.000000,0.000000



--- FINAL BASELINE EVALUATION SCORES ---
Precision: 0.0000
Recall: 0.0000
BASELINE F1 SCORE: 0.0000


In [13]:
import os
backup_path = '/content/drive/MyDrive/MedScriptAI-checkpoints/mbert-baseline'
os.makedirs(backup_path, exist_ok=True)
trainer.save_model(backup_path)
print("Success! Model weights permanently backed up via UI mount.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Success! Model weights permanently backed up via UI mount.


In [14]:
from transformers import AutoTokenizer

# 1. Initialize the XLM-RoBERTa base tokenizer engine
XLM_MODEL_NAME = 'xlm-roberta-base'
xlm_tokenizer = AutoTokenizer.from_pretrained(XLM_MODEL_NAME)
print('XLM-RoBERTa Tokenizer loaded. Vocab size:', xlm_tokenizer.vocab_size)

# 2. Define the subword alignment function specifically for XLM-R
def tokenize_and_align_labels_xlm(examples):
    tokenized = xlm_tokenizer(examples['tokens'], truncation=True, is_split_into_words=True, padding='max_length', max_length=128)
    labels = []
    for i, label_list in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100) # Mask special tokens
            elif word_id != prev_word_id:
                label_ids.append(LABEL2ID.get(label_list[word_id], 0))
            else:
                label_ids.append(-100) # Mask sub-words split by tokenizer
            prev_word_id = word_id
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

# 3. Map the datasets using the XLM-R tokenization process
xlm_tokenized_datasets = hf_dataset.map(tokenize_and_align_labels_xlm, batched=True)
print("XLM-RoBERTa Data mapping complete. Ready for Step 6.2 training!")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

XLM-RoBERTa Tokenizer loaded. Vocab size: 250002


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

XLM-RoBERTa Data mapping complete. Ready for Step 6.2 training!


In [15]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np

# 1. Initialize XLM-RoBERTa for Token Classification
xlm_model = AutoModelForTokenClassification.from_pretrained(
    XLM_MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# 2. Establish training hyper-parameters matching project specifications
xlm_training_args = TrainingArguments(
    output_dir='./xlmr-fine-tuned',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy='epoch',     # Using the updated HF parameter
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none'
)

# 3. Initialize Trainer wrapper engine for XLM-R
xlm_trainer = Trainer(
    model=xlm_model,
    args=xlm_training_args,
    train_dataset=xlm_tokenized_datasets['train'],
    eval_dataset=xlm_tokenized_datasets['validation'],
    compute_metrics=compute_metrics  # Reuses our verified seqeval evaluation function
)

# 4. Start fine-tuning process on GPU
print("Launching XLM-RoBERTa model training run...")
xlm_trainer.train()

# 5. Extract test split evaluations
xlm_results = xlm_trainer.evaluate(xlm_tokenized_datasets['test'])
print("\n--- FINAL XLM-ROBERTA EVALUATION SCORES ---")
print(f"Precision: {xlm_results['eval_precision']:.4f}")
print(f"Recall: {xlm_results['eval_recall']:.4f}")
print(f"XLM-RoBERTa F1 SCORE: {xlm_results['eval_f1']:.4f}")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Launching XLM-RoBERTa model training run...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,2.465460,0.000000,0.000000,0.000000
2,No log,2.430236,0.000000,0.000000,0.000000
3,No log,2.413468,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Precision,Recall,F1
No log,2.395024,3,0.000000,0.000000,0.000000



--- FINAL XLM-ROBERTA EVALUATION SCORES ---
Precision: 0.0000
Recall: 0.0000
XLM-RoBERTa F1 SCORE: 0.0000


In [16]:
import os

# Create a separate backup directory for XLM-R
xlm_backup_path = '/content/drive/MyDrive/MedScriptAI-checkpoints/xlmr-fine-tuned'
os.makedirs(xlm_backup_path, exist_ok=True)

# Save the model state
xlm_trainer.save_model(xlm_backup_path)
print(f"Success! XLM-RoBERTa weights permanently backed up to Drive at: {xlm_backup_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Success! XLM-RoBERTa weights permanently backed up to Drive at: /content/drive/MyDrive/MedScriptAI-checkpoints/xlmr-fine-tuned


In [17]:
import pandas as pd

# 1. Gather recorded evaluation metrics from both sessions
comparison_data = {
    "Model Architecture": ["mBERT Baseline", "XLM-RoBERTa Fine-Tuned"],
    "Precision": [baseline_results.get('eval_precision', 0.0), xlm_results.get('eval_precision', 0.0)],
    "Recall": [baseline_results.get('eval_recall', 0.0), xlm_results.get('eval_recall', 0.0)],
    "F1-Score": [baseline_results.get('eval_f1', 0.0), xlm_results.get('eval_f1', 0.0)]
}

# 2. Build structured data frame view
df_metrics = pd.DataFrame(comparison_data)

print("\n=======================================================")
print("          MEDSCRIPT-AI MODEL COMPARISON DASHBOARD       ")
print("=======================================================\n")
print(df_metrics.to_string(index=False))
print("\n=======================================================")


          MEDSCRIPT-AI MODEL COMPARISON DASHBOARD       

    Model Architecture  Precision  Recall  F1-Score
        mBERT Baseline        0.0     0.0       0.0
XLM-RoBERTa Fine-Tuned        0.0     0.0       0.0



In [18]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# 1. Define the path to your best saved model (XLM-RoBERTa)
saved_model_path = '/content/drive/MyDrive/MedScriptAI-checkpoints/xlmr-fine-tuned'

# 2. Load the fine-tuned tokenizer and model directly from your Drive
print("Loading fine-tuned model from Drive...")
loaded_tokenizer = AutoTokenizer.from_pretrained(saved_model_path)
loaded_model = AutoModelForTokenClassification.from_pretrained(saved_model_path)

# 3. Initialize the Hugging Face NER Pipeline
# We use aggregation_strategy="simple" to automatically stitch B- and I- tags back together!
ner_pipeline = pipeline(
    "ner",
    model=loaded_model,
    tokenizer=loaded_tokenizer,
    aggregation_strategy="simple"
)

# 4. Test it on unseen, raw code-switched text
test_sentence = "مریض کو severe headache اور fever ہے. Prescribe Panadol 500mg oral twice daily."

print("\n=======================================================")
print(f"INPUT TEXT: {test_sentence}")
print("=======================================================\n")
print("--- EXTRACTED MEDICAL ENTITIES ---")

# 5. Run the prediction and format the results
predictions = ner_pipeline(test_sentence)

for prediction in predictions:
    # Clean up XLM-R's special ' ' token character for cleaner display
    word = prediction['word'].replace(' ', '')
    label = prediction['entity_group']
    confidence = prediction['score']

    print(f"Entity: {word:<15} | Label: {label:<10} | Confidence: {confidence:.4f}")

Loading fine-tuned model from Drive...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


INPUT TEXT: مریض کو severe headache اور fever ہے. Prescribe Panadol 500mg oral twice daily.

--- EXTRACTED MEDICAL ENTITIES ---
Entity: مریض            | Label: DRUG       | Confidence: 0.1320
Entity: کو              | Label: DRUG       | Confidence: 0.1267
Entity: severe          | Label: DRUG       | Confidence: 0.1237
Entity: headacheاورfever | Label: DOSE       | Confidence: 0.1228
Entity: ہے.             | Label: DRUG       | Confidence: 0.1251
Entity: Prescribe       | Label: DRUG       | Confidence: 0.1277
Entity: Panadol         | Label: DRUG       | Confidence: 0.1293
Entity: 500mg           | Label: DRUG       | Confidence: 0.1289
Entity: oral            | Label: DRUG       | Confidence: 0.1271
Entity: twice           | Label: DRUG       | Confidence: 0.1241
Entity: daily.          | Label: DRUG       | Confidence: 0.1246


In [19]:
import json

# 1. Define a list of raw clinical strings simulating mixed doctor notes
batch_prescriptions = [
    "Patient ko acute stomach pain bleeding ho rahi hai. Give Flagyl 400mg instantly.",
    "مریض کو high blood pressure ہے، prescribe Capoten 25mg oral sublingual.",
    "Follow up after 2 weeks for diabetes checkup. Continue Insulin 10 units before breakfast."
]

def process_batch_inference(texts):
    batch_results = []

    # Run text through our active pipeline
    pipeline_outputs = ner_pipeline(texts)

    # Process each sentence output along with its source text
    for text, entities in zip(texts, pipeline_outputs):
        extracted_entities = []
        for ent in entities:
            extracted_entities.append({
                "entity": ent['word'].replace(' ', ''),
                "label": ent['entity_group'],
                "confidence": float(ent['score']),
                "start_char": ent['start'],
                "end_char": ent['end']
            })

        batch_results.append({
            "raw_text": text,
            "extracted_medical_entities": extracted_entities
        })
    return batch_results

# 2. Run the processing engine
print("Processing batch inference on clinical data...")
structured_outputs = process_batch_inference(batch_prescriptions)

# 3. Export structured records safely to local disk workspace
output_file = 'structured_clinical_entities.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(structured_outputs, f, ensure_ascii=False, indent=2)

print(f"\nSuccess! Structured entities compiled and saved to: {output_file}")
print("Printing first parsed record sample:\n")
print(json.dumps(structured_outputs[0], indent=2))

Processing batch inference on clinical data...

Success! Structured entities compiled and saved to: structured_clinical_entities.json
Printing first parsed record sample:

{
  "raw_text": "Patient ko acute stomach pain bleeding ho rahi hai. Give Flagyl 400mg instantly.",
  "extracted_medical_entities": [
    {
      "entity": "Patient",
      "label": "DRUG",
      "confidence": 0.1319819688796997,
      "start_char": 0,
      "end_char": 7
    },
    {
      "entity": "ko",
      "label": "DRUG",
      "confidence": 0.126695454120636,
      "start_char": 8,
      "end_char": 10
    },
    {
      "entity": "acute",
      "label": "DRUG",
      "confidence": 0.12374036759138107,
      "start_char": 11,
      "end_char": 16
    },
    {
      "entity": "stomachpainbleeding",
      "label": "DOSE",
      "confidence": 0.1228121891617775,
      "start_char": 17,
      "end_char": 38
    },
    {
      "entity": "ho",
      "label": "DRUG",
      "confidence": 0.1251237690448761,
      "st